# 05 - Correlaciones en Profundidad: Relaciones Verdaderas vs Espurias

## Fase 3: Estadistica Inferencial — Proyecto NYC Taxi

En este notebook exploramos las correlaciones entre variables de los viajes en taxi, utilizando tecnicas avanzadas para distinguir relaciones reales de asociaciones espurias.

### Pregunta de negocio
**Que variables estan realmente relacionadas entre si?**

### Objetivos de aprendizaje
- Comparar **correlacion de Pearson** (lineal) con **Spearman** (monotona)
- Aplicar **correlacion parcial** para controlar variables confusoras
- Analizar como las correlaciones cambian por **contexto** (hora, zona)
- Usar **correlacion point-biserial** para variables binarias
- Distinguir entre correlaciones verdaderas y espurias

### Concepto clave
Dos variables pueden estar correlacionadas sin tener una relacion causal directa. La **correlacion parcial** y el analisis contextual nos ayudan a separar las relaciones genuinas de las que surgen por variables confusoras.

---
## 1. Carga de datos y preparacion

In [ ]:
import sys
sys.path.insert(0, '../../../src')
from bigquery.bq_helper import BigQueryHelper
bq = BigQueryHelper()

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings

warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

In [ ]:
query = """
SELECT
    pickup_datetime,
    dropoff_datetime,
    passenger_count,
    trip_distance,
    fare_amount,
    tip_amount,
    tolls_amount,
    total_amount,
    payment_type,
    pickup_location_id,
    EXTRACT(HOUR FROM pickup_datetime) AS pickup_hour,
    EXTRACT(DAYOFWEEK FROM pickup_datetime) AS day_of_week,
    TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, MINUTE) AS trip_duration_min,
    -- Variables derivadas
    SAFE_DIVIDE(fare_amount, trip_distance) AS fare_per_mile,
    SAFE_DIVIDE(tip_amount, fare_amount) * 100 AS tip_percentage,
    SAFE_DIVIDE(trip_distance, 
        TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, MINUTE) / 60.0) AS speed_mph,
    -- Zona de recogida basada en pickup_location_id (TLC Zone IDs)
    CASE
        WHEN pickup_location_id IN ('4','12','13','24','41','42','43','45','48','50','68','74','75','79','87','88','90','100','107','113','114','116','125','127','128','137','140','141','142','143','144','148','151','152','153','158','161','162','163','164','166','170','186','194','202','209','211','224','229','230','231','232','233','234','236','237','238','239','243','244','246','249','261','262','263')
        THEN 'Manhattan'
        WHEN pickup_location_id IN ('11','14','17','21','22','25','26','29','33','34','35','36','37','39','40','49','52','54','55','61','62','63','65','66','67','69','71','72','76','77','80','85','89','91','97','106','108','111','112','123','133','149','150','154','155','165','177','178','181','188','189','190','195','210','217','222','225','227','228','255','256','257')
        THEN 'Brooklyn'
        WHEN pickup_location_id = '132'
        THEN 'JFK_Airport'
        WHEN pickup_location_id = '138'
        THEN 'LaGuardia_Airport'
        ELSE 'Otra'
    END AS pickup_zone,
    -- Banda horaria
    CASE
        WHEN EXTRACT(HOUR FROM pickup_datetime) BETWEEN 6 AND 11 THEN 'Manana'
        WHEN EXTRACT(HOUR FROM pickup_datetime) BETWEEN 12 AND 17 THEN 'Tarde'
        WHEN EXTRACT(HOUR FROM pickup_datetime) BETWEEN 18 AND 22 THEN 'Noche'
        ELSE 'Madrugada'
    END AS hour_band,
    -- Variables binarias
    CASE WHEN EXTRACT(DAYOFWEEK FROM pickup_datetime) IN (1, 7) THEN 1 ELSE 0 END AS is_weekend,
    CASE
        WHEN EXTRACT(HOUR FROM pickup_datetime) BETWEEN 7 AND 9
          OR EXTRACT(HOUR FROM pickup_datetime) BETWEEN 17 AND 19
        THEN 1 ELSE 0
    END AS is_rush_hour
FROM `bigquery-public-data.new_york_taxi_trips.tlc_yellow_trips_2015`
WHERE
    fare_amount BETWEEN 2.5 AND 200
    AND trip_distance BETWEEN 0.1 AND 50
    AND passenger_count BETWEEN 1 AND 6
    AND pickup_location_id IS NOT NULL
    AND TIMESTAMP_DIFF(dropoff_datetime, pickup_datetime, MINUTE) BETWEEN 1 AND 180
ORDER BY FARM_FINGERPRINT(CAST(pickup_datetime AS STRING) || CAST(dropoff_datetime AS STRING))
LIMIT 50000
"""

df = bq.run_query(query, label='phase3_05_correlations')
print(f"Registros cargados: {len(df):,}")
df.head()

In [ ]:
# Filtrar valores extremos en variables derivadas
df = df[(df['fare_per_mile'] > 0) & (df['fare_per_mile'] < 30)].copy()
df = df[(df['speed_mph'] > 0) & (df['speed_mph'] < 80)].copy()
df = df[(df['tip_percentage'] >= 0) & (df['tip_percentage'] < 100)].copy()

# Definir columnas numericas para el analisis
numeric_cols = ['trip_distance', 'fare_amount', 'tip_amount', 'tolls_amount',
                'total_amount', 'trip_duration_min', 'passenger_count',
                'fare_per_mile', 'tip_percentage', 'speed_mph']

print(f"Registros despues de filtrar: {len(df):,}")
print(f"\nResumen de variables numericas:")
df[numeric_cols].describe().round(2)

---
## 2. Matriz de correlacion de Pearson

La **correlacion de Pearson** mide la fuerza de la relacion **lineal** entre dos variables. Asume:
- Variables continuas
- Relacion lineal
- Distribucion aproximadamente normal

Su coeficiente $r$ va de -1 a +1:
- $|r| > 0.7$: correlacion fuerte
- $0.3 < |r| < 0.7$: correlacion moderada
- $|r| < 0.3$: correlacion debil

In [ ]:
# Matriz de correlacion de Pearson
corr_pearson = df[numeric_cols].corr(method='pearson')

# Heatmap
fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_pearson, dtype=bool))

sns.heatmap(corr_pearson.astype(float), mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',            center=0, vmin=-1, vmax=1, linewidths=0.5, square=True, ax=ax,
            cbar_kws={'label': 'Correlacion de Pearson'})

ax.set_title('Matriz de Correlacion de Pearson — Todas las variables numericas', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Top correlaciones
corr_pairs = []
for i in range(len(numeric_cols)):
    for j in range(i+1, len(numeric_cols)):
        corr_pairs.append({
            'Variable 1': numeric_cols[i],
            'Variable 2': numeric_cols[j],
            'Pearson r': corr_pearson.iloc[i, j]
        })

df_pairs = pd.DataFrame(corr_pairs)
df_pairs['|r|'] = df_pairs['Pearson r'].abs()
df_pairs = df_pairs.sort_values('|r|', ascending=False)

print("Top 15 correlaciones de Pearson (por magnitud):")
print(df_pairs.head(15).to_string(index=False))

---
## 3. Matriz de correlacion de Spearman

La **correlacion de Spearman** mide la fuerza de la relacion **monotona** (no necesariamente lineal) entre dos variables. Trabaja con los **rangos** de los datos, no con los valores directos.

### Cuando usar cada una?

| Criterio | Pearson | Spearman |
|----------|---------|----------|
| Tipo de relacion | Lineal | Monotona (lineal o no) |
| Sensibilidad a outliers | Alta | Baja |
| Supuesto de normalidad | Si | No |
| Variables ordinales | No adecuado | Adecuado |

Si Spearman > Pearson para un par de variables, sugiere que la relacion existe pero **no es lineal**.

In [ ]:
# Matriz de correlacion de Spearman
corr_spearman = df[numeric_cols].corr(method='spearman')

# Heatmap
fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(corr_spearman, dtype=bool))

sns.heatmap(corr_spearman.astype(float), mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',            center=0, vmin=-1, vmax=1, linewidths=0.5, square=True, ax=ax,
            cbar_kws={'label': 'Correlacion de Spearman'})

ax.set_title('Matriz de Correlacion de Spearman — Todas las variables numericas', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Comparacion Pearson vs Spearman: donde difieren mas?
diff_matrix = (corr_spearman - corr_pearson).abs()

fig, ax = plt.subplots(figsize=(14, 11))
mask = np.triu(np.ones_like(diff_matrix, dtype=bool))

sns.heatmap(diff_matrix.astype(float), mask=mask, annot=True, fmt='.3f', cmap='Oranges',            linewidths=0.5, square=True, ax=ax,
            cbar_kws={'label': '|Spearman - Pearson|'})

ax.set_title('Diferencia |Spearman - Pearson| — Donde la relacion NO es lineal', fontsize=14, pad=20)
plt.tight_layout()
plt.show()

In [ ]:
# Pares con mayor diferencia entre Pearson y Spearman
diff_pairs = []
for i in range(len(numeric_cols)):
    for j in range(i+1, len(numeric_cols)):
        diff_pairs.append({
            'Variable 1': numeric_cols[i],
            'Variable 2': numeric_cols[j],
            'Pearson': corr_pearson.iloc[i, j],
            'Spearman': corr_spearman.iloc[i, j],
            'Diferencia': abs(corr_spearman.iloc[i, j] - corr_pearson.iloc[i, j])
        })

df_diff = pd.DataFrame(diff_pairs).sort_values('Diferencia', ascending=False)
print("Pares donde Pearson y Spearman difieren mas (relacion no lineal):")
print(df_diff.head(10).round(3).to_string(index=False))

---
## 4. Correlacion parcial: Distancia-Tarifa controlando por duracion

La **correlacion parcial** mide la relacion entre dos variables **eliminando el efecto** de una o mas variables confusoras.

### Pregunta
Sabemos que `trip_distance` y `fare_amount` estan muy correlacionadas. Pero ambas tambien se correlacionan con `trip_duration_min`. Si controlamos por la duracion del viaje, se mantiene la correlacion distancia-tarifa?

### Metodo
Calculamos la correlacion parcial usando residuos de regresion:
1. Regresamos `trip_distance` contra `trip_duration_min` -> obtenemos residuos
2. Regresamos `fare_amount` contra `trip_duration_min` -> obtenemos residuos
3. Correlacionamos los residuos -> esa es la correlacion parcial

In [ ]:
def partial_correlation(df, x, y, covariates):
    """
    Calcula la correlacion parcial entre x e y controlando por covariates.
    Usa el metodo de residuos de regresion lineal.
    
    Parameters
    ----------
    df : DataFrame
    x, y : str - nombres de columnas
    covariates : list of str - variables a controlar
    
    Returns
    -------
    dict con r_parcial, p_valor, r_simple
    """
    from numpy.linalg import lstsq
    
    data = df[[x, y] + covariates].dropna()
    
    # Correlacion simple (sin controlar)
    r_simple, _ = stats.pearsonr(data[x], data[y])
    
    # Matriz de covariables con intercepto
    Z = np.column_stack([np.ones(len(data))] + [data[c].values for c in covariates])
    
    # Residuos de x ~ covariables
    beta_x, _, _, _ = lstsq(Z, data[x].values, rcond=None)
    resid_x = data[x].values - Z @ beta_x
    
    # Residuos de y ~ covariables
    beta_y, _, _, _ = lstsq(Z, data[y].values, rcond=None)
    resid_y = data[y].values - Z @ beta_y
    
    # Correlacion de residuos = correlacion parcial
    r_parcial, p_valor = stats.pearsonr(resid_x, resid_y)
    
    return {
        'r_simple': r_simple,
        'r_parcial': r_parcial,
        'p_valor': p_valor,
        'cambio': r_simple - r_parcial,
        'cambio_pct': (1 - abs(r_parcial) / abs(r_simple)) * 100 if r_simple != 0 else 0
    }

In [ ]:
# Correlacion parcial: distancia-tarifa controlando por duracion
result_df = partial_correlation(df, 'trip_distance', 'fare_amount', ['trip_duration_min'])

print("Correlacion parcial: trip_distance ~ fare_amount | trip_duration_min")
print(f"{'='*65}")
print(f"Correlacion simple (Pearson):  r = {result_df['r_simple']:.4f}")
print(f"Correlacion parcial:           r = {result_df['r_parcial']:.4f}")
print(f"p-valor:                       {result_df['p_valor']:.2e}")
print(f"Reduccion:                     {result_df['cambio_pct']:.1f}%")
print()
print("Interpretacion: Al controlar por duracion del viaje,")
if abs(result_df['r_parcial']) > 0.3:
    print("la correlacion distancia-tarifa SE MANTIENE fuerte.")
    print("La distancia tiene un efecto directo sobre la tarifa,")
    print("independiente de la duracion.")
else:
    print("la correlacion se reduce sustancialmente.")
    print("Gran parte de la relacion estaba mediada por la duracion.")

In [ ]:
# Visualizacion: correlacion simple vs parcial
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

sample = df.sample(n=3000, random_state=42)

# 1. Correlacion simple distancia-tarifa
axes[0].scatter(sample['trip_distance'], sample['fare_amount'],
                alpha=0.1, s=5, color='steelblue')
axes[0].set_xlabel('Distancia (millas)')
axes[0].set_ylabel('Tarifa (USD)')
axes[0].set_title(f'Simple: r = {result_df["r_simple"]:.3f}')
axes[0].set_xlim(0, 25)
axes[0].set_ylim(0, 80)

# 2. Ambas vs duracion (la confusora)
axes[1].scatter(sample['trip_duration_min'], sample['trip_distance'],
                alpha=0.1, s=5, color='green', label='Distancia')
ax2 = axes[1].twinx()
ax2.scatter(sample['trip_duration_min'], sample['fare_amount'],
            alpha=0.1, s=5, color='orange', label='Tarifa')
axes[1].set_xlabel('Duracion (min)')
axes[1].set_ylabel('Distancia (millas)', color='green')
ax2.set_ylabel('Tarifa (USD)', color='orange')
axes[1].set_title('Ambas dependen de la duracion')
axes[1].set_xlim(0, 60)

# 3. Residuos (correlacion parcial)
from numpy.linalg import lstsq
data_vis = sample[['trip_distance', 'fare_amount', 'trip_duration_min']].dropna()
Z = np.column_stack([np.ones(len(data_vis)), data_vis['trip_duration_min'].values])
beta_x, _, _, _ = lstsq(Z, data_vis['trip_distance'].values, rcond=None)
beta_y, _, _, _ = lstsq(Z, data_vis['fare_amount'].values, rcond=None)
resid_x = data_vis['trip_distance'].values - Z @ beta_x
resid_y = data_vis['fare_amount'].values - Z @ beta_y

axes[2].scatter(resid_x, resid_y, alpha=0.1, s=5, color='purple')
axes[2].set_xlabel('Residuos de distancia')
axes[2].set_ylabel('Residuos de tarifa')
axes[2].set_title(f'Parcial: r = {result_df["r_parcial"]:.3f}')
axes[2].axhline(0, color='gray', linestyle='--', alpha=0.3)
axes[2].axvline(0, color='gray', linestyle='--', alpha=0.3)

plt.tight_layout()
plt.show()

---
## 5. Correlacion parcial: Propina-Duracion controlando por tarifa

Los pasajeros dan mas propina en viajes largos? O simplemente dan mas propina porque el viaje cuesta mas?

### Hipotesis
- Si la correlacion `tip_percentage ~ trip_duration_min` desaparece al controlar por `fare_amount`, entonces la propina depende del monto, no de la duracion en si.

In [ ]:
# Solo viajes con propina registrada (tarjeta)
df_tip = df[df['tip_amount'] > 0].copy()

# Correlacion parcial: tip_percentage ~ trip_duration_min | fare_amount
result_tip = partial_correlation(df_tip, 'tip_percentage', 'trip_duration_min', ['fare_amount'])

print("Correlacion parcial: tip_percentage ~ trip_duration_min | fare_amount")
print(f"{'='*65}")
print(f"Correlacion simple:   r = {result_tip['r_simple']:.4f}")
print(f"Correlacion parcial:  r = {result_tip['r_parcial']:.4f}")
print(f"p-valor:              {result_tip['p_valor']:.2e}")
print(f"Reduccion:            {result_tip['cambio_pct']:.1f}%")
print()

# Tambien: tip_amount ~ trip_duration_min | fare_amount
result_tip_abs = partial_correlation(df_tip, 'tip_amount', 'trip_duration_min', ['fare_amount'])

print("Correlacion parcial: tip_amount ~ trip_duration_min | fare_amount")
print(f"{'='*65}")
print(f"Correlacion simple:   r = {result_tip_abs['r_simple']:.4f}")
print(f"Correlacion parcial:  r = {result_tip_abs['r_parcial']:.4f}")
print(f"Reduccion:            {result_tip_abs['cambio_pct']:.1f}%")

In [ ]:
# Tabla resumen de correlaciones parciales
partial_tests = [
    ('trip_distance', 'fare_amount', ['trip_duration_min'], 'Distancia-Tarifa | Duracion'),
    ('tip_percentage', 'trip_duration_min', ['fare_amount'], 'Propina%-Duracion | Tarifa'),
    ('tip_amount', 'trip_distance', ['fare_amount'], 'Propina$-Distancia | Tarifa'),
    ('fare_per_mile', 'trip_duration_min', ['trip_distance'], 'Tarifa/milla-Duracion | Distancia'),
    ('speed_mph', 'fare_amount', ['trip_distance'], 'Velocidad-Tarifa | Distancia'),
]

results_table = []
for x, y, covs, label in partial_tests:
    r = partial_correlation(df, x, y, covs)
    results_table.append({
        'Relacion': label,
        'r simple': round(r['r_simple'], 3),
        'r parcial': round(r['r_parcial'], 3),
        'Reduccion %': round(r['cambio_pct'], 1),
        'Genuina?': 'Si' if abs(r['r_parcial']) > 0.15 else 'Debil/Espuria'
    })

df_results = pd.DataFrame(results_table)
print("Resumen de correlaciones parciales:")
print(df_results.to_string(index=False))

---
## 6. Correlaciones por banda horaria

Las relaciones entre variables no son estaticas: pueden **cambiar segun el contexto**. Analizamos como las correlaciones principales varian segun el momento del dia.

### Por que es importante?
Si la correlacion distancia-tarifa cambia por hora, puede indicar que factores como el trafico (horas pico) o los recargos nocturnos modifican la estructura de precios.

In [ ]:
# Correlaciones clave por banda horaria
key_pairs = [
    ('trip_distance', 'fare_amount'),
    ('trip_duration_min', 'fare_amount'),
    ('trip_distance', 'tip_amount'),
    ('fare_amount', 'tip_percentage'),
    ('speed_mph', 'fare_per_mile'),
]

hour_order = ['Manana', 'Tarde', 'Noche', 'Madrugada']
corr_by_hour = []

for band in hour_order:
    subset = df[df['hour_band'] == band]
    for v1, v2 in key_pairs:
        r, p = stats.pearsonr(subset[v1].dropna(), subset[v2].dropna())
        corr_by_hour.append({
            'Banda': band,
            'Par': f'{v1} ~ {v2}',
            'r': r,
            'n': len(subset)
        })

df_hour_corr = pd.DataFrame(corr_by_hour)
pivot_hour = df_hour_corr.pivot(index='Par', columns='Banda', values='r')[hour_order]

print("Correlaciones de Pearson por banda horaria:")
print(pivot_hour.round(3))

In [ ]:
# Heatmap de correlaciones por banda horaria
fig, ax = plt.subplots(figsize=(12, 6))

sns.heatmap(pivot_hour.astype(float), annot=True, fmt='.3f', cmap='RdBu_r',            center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax,
            cbar_kws={'label': 'Correlacion de Pearson'})

ax.set_title('Como cambian las correlaciones segun la hora del dia', fontsize=14, pad=15)
ax.set_xlabel('Banda horaria')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

In [ ]:
# Scatter por banda horaria para el par distancia-tarifa
fig, axes = plt.subplots(1, 4, figsize=(20, 5), sharey=True)
colors = ['#FF9800', '#2196F3', '#9C27B0', '#607D8B']

for ax, band, color in zip(axes, hour_order, colors):
    subset = df[df['hour_band'] == band].sample(n=min(2000, len(df[df['hour_band'] == band])),
                                                 random_state=42)
    r = subset['trip_distance'].corr(subset['fare_amount'])
    
    sns.regplot(data=subset, x='trip_distance', y='fare_amount',
                scatter_kws={'alpha': 0.1, 's': 5, 'color': color},
                line_kws={'color': 'red', 'linewidth': 2},
                ci=95, ax=ax)
    ax.set_xlim(0, 20)
    ax.set_ylim(0, 60)
    ax.set_title(f'{band}\nr = {r:.3f} (n={len(subset):,})')
    ax.set_xlabel('Distancia (millas)')

axes[0].set_ylabel('Tarifa (USD)')
fig.suptitle('Relacion distancia-tarifa por banda horaria', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 7. Correlaciones por zona geografica

Las relaciones tambien pueden variar por ubicacion. Manhattan con su trafico denso puede tener patrones distintos a los aeropuertos con sus tarifas fijas.

In [ ]:
# Correlaciones clave por zona
zones_analysis = ['Manhattan', 'Brooklyn', 'Queens_West', 'JFK_Airport', 'LaGuardia_Airport']

corr_by_zone = []
for zone in zones_analysis:
    subset = df[df['pickup_zone'] == zone]
    if len(subset) < 50:
        continue
    for v1, v2 in key_pairs:
        data = subset[[v1, v2]].dropna()
        if len(data) < 30:
            continue
        r, p = stats.pearsonr(data[v1], data[v2])
        corr_by_zone.append({
            'Zona': zone,
            'Par': f'{v1} ~ {v2}',
            'r': r,
            'n': len(data)
        })

df_zone_corr = pd.DataFrame(corr_by_zone)
if len(df_zone_corr) > 0:
    pivot_zone = df_zone_corr.pivot(index='Par', columns='Zona', values='r')
    # Reordenar columnas
    available_zones = [z for z in zones_analysis if z in pivot_zone.columns]
    pivot_zone = pivot_zone[available_zones]
    
    print("Correlaciones de Pearson por zona de recogida:")
    print(pivot_zone.round(3))

In [ ]:
# Heatmap de correlaciones por zona
if len(df_zone_corr) > 0:
    fig, ax = plt.subplots(figsize=(14, 6))

    sns.heatmap(pivot_zone.astype(float), annot=True, fmt='.3f', cmap='RdBu_r',                center=0, vmin=-1, vmax=1, linewidths=0.5, ax=ax,
                cbar_kws={'label': 'Correlacion de Pearson'})

    ax.set_title('Variacion geografica de las correlaciones', fontsize=14, pad=15)
    ax.set_xlabel('Zona de recogida')
    ax.set_ylabel('')
    plt.tight_layout()
    plt.show()
else:
    print("No hay suficientes datos para todas las zonas.")

**Interpretacion de la variacion geografica:**
- Los **aeropuertos** (JFK, LaGuardia) pueden mostrar correlaciones distintas debido a las tarifas fijas
- **Manhattan** puede tener una relacion distancia-tarifa diferente por el efecto del trafico (mas tiempo por menos distancia)
- Las zonas con menos datos tendran correlaciones menos estables

---
## 8. Scatter matrix con lineas de regresion para los pares mas correlacionados

Visualizamos los pares de variables con las correlaciones mas fuertes usando scatter plots con lineas de regresion superpuestas.

In [ ]:
# Top 6 pares mas correlacionados (excluyendo pares triviales como total_amount ~ fare_amount)
trivial_pairs = [
    ('fare_amount', 'total_amount'),
    ('total_amount', 'fare_amount'),
    ('tip_amount', 'total_amount'),
    ('total_amount', 'tip_amount'),
]

top_pairs = df_pairs[
    ~df_pairs.apply(lambda r: (r['Variable 1'], r['Variable 2']) in trivial_pairs, axis=1)
].head(6)

print("Top 6 pares correlacionados (no triviales):")
print(top_pairs[['Variable 1', 'Variable 2', 'Pearson r']].to_string(index=False))

In [ ]:
# Scatter matrix con regresion
fig, axes = plt.subplots(2, 3, figsize=(18, 12))
sample_vis = df.sample(n=3000, random_state=42)

for idx, (_, row) in enumerate(top_pairs.iterrows()):
    ax = axes.flat[idx]
    v1, v2 = row['Variable 1'], row['Variable 2']
    r = row['Pearson r']
    
    sns.regplot(data=sample_vis, x=v1, y=v2,
                scatter_kws={'alpha': 0.1, 's': 8},
                line_kws={'color': 'red', 'linewidth': 2},
                ci=95, ax=ax)
    ax.set_title(f'{v1} vs {v2}\nr = {r:.3f}', fontsize=11)
    ax.set_xlabel(v1)
    ax.set_ylabel(v2)

fig.suptitle('Top 6 pares correlacionados (con linea de regresion)', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 9. Correlacion point-biserial para variables binarias

La **correlacion point-biserial** es un caso especial de la correlacion de Pearson donde una variable es continua y la otra es **binaria** (0/1). Es matematicamente equivalente a Pearson, pero conceptualmente adecuada para este tipo de datos.

Tenemos dos variables binarias:
- `is_weekend` (0 = dia laboral, 1 = fin de semana)
- `is_rush_hour` (0 = no hora pico, 1 = hora pico)

In [ ]:
# Correlacion point-biserial para variables binarias
binary_vars = ['is_weekend', 'is_rush_hour']
continuous_vars = ['fare_amount', 'trip_distance', 'trip_duration_min',
                   'tip_percentage', 'speed_mph', 'fare_per_mile']

pb_results = []
for bvar in binary_vars:
    for cvar in continuous_vars:
        data = df[[bvar, cvar]].dropna()
        r, p = stats.pointbiserialr(data[bvar], data[cvar])
        
        # Medias por grupo
        mean_0 = data[data[bvar] == 0][cvar].mean()
        mean_1 = data[data[bvar] == 1][cvar].mean()
        
        pb_results.append({
            'Binaria': bvar,
            'Continua': cvar,
            'r_pb': round(r, 4),
            'p-valor': f"{p:.2e}",
            'Media(0)': round(mean_0, 2),
            'Media(1)': round(mean_1, 2),
            'Significativo': 'Si' if p < 0.05 else 'No'
        })

df_pb = pd.DataFrame(pb_results)
print("Correlacion point-biserial:")
print(df_pb.to_string(index=False))

In [ ]:
# Visualizacion: box plots para las correlaciones point-biserial mas fuertes
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for idx, cvar in enumerate(continuous_vars):
    ax = axes.flat[idx]
    
    # Crear DataFrame para plot
    plot_data = df[['is_rush_hour', 'is_weekend', cvar]].melt(
        id_vars=[cvar], var_name='Variable binaria', value_name='Valor'
    )
    plot_data['Grupo'] = plot_data.apply(
        lambda r: f"{r['Variable binaria']}={int(r['Valor'])}", axis=1
    )
    
    sns.boxplot(data=df, x='is_rush_hour', y=cvar, ax=ax,
                palette=['#4CAF50', '#F44336'], showfliers=False)
    ax.set_xlabel('Hora pico (0=No, 1=Si)')
    ax.set_ylabel(cvar)
    ax.set_title(f'{cvar} por hora pico')

plt.suptitle('Efecto de hora pico en variables del viaje', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Lo mismo para fin de semana
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for idx, cvar in enumerate(continuous_vars):
    ax = axes.flat[idx]
    sns.boxplot(data=df, x='is_weekend', y=cvar, ax=ax,
                palette=['#2196F3', '#FF9800'], showfliers=False)
    ax.set_xlabel('Fin de semana (0=No, 1=Si)')
    ax.set_ylabel(cvar)
    ax.set_title(f'{cvar} por dia de semana')

plt.suptitle('Efecto de fin de semana en variables del viaje', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

---
## 10. Resumen: Correlaciones verdaderas vs espurias

### Hallazgos principales

#### Correlaciones genuinas (se mantienen al controlar confusoras)
- **Distancia - Tarifa:** La relacion mas fuerte y directa. Se mantiene al controlar por duracion, confirmando que la tarifa depende fundamentalmente de la distancia recorrida.
- **Duracion - Tarifa:** Tambien genuina, refleja el componente temporal del taximetro.

#### Correlaciones posiblemente espurias o mediadas
- **Tip amount - Distancia:** Gran parte de esta correlacion puede estar mediada por la tarifa (viajes largos = tarifa alta = propina alta en monto absoluto).
- **Velocidad - Tarifa por milla:** Puede ser un artefacto: baja velocidad (trafico) = mas tiempo por milla = mayor costo por milla.

#### Variacion contextual
- Las correlaciones **cambian por hora del dia**: el trafico en hora pico modifica las relaciones.
- Las correlaciones **cambian por zona**: los aeropuertos con tarifas fijas rompen los patrones lineales habituales.

### Leccion metodologica

| Tecnica | Para que sirve | Cuando usarla |
|---------|---------------|---------------|
| Pearson | Relacion lineal entre continuas | Variables con distribucion normal, relacion lineal |
| Spearman | Relacion monotona | Outliers, no normalidad, relacion no lineal |
| Parcial | Controlar confusoras | Sospecha de variable mediadora o confusora |
| Point-biserial | Una binaria, una continua | Variables dicotomicas como es_fin_de_semana |
| Por contexto | Detectar heterogeneidad | Sospecha de que la relacion cambia por grupo |

### Implicaciones para el modelado (proxima fase)

1. **Variables predictoras fuertes:** `trip_distance` y `trip_duration_min` son candidatos clave para modelos de prediccion de tarifa.
2. **Cuidado con la colinealidad:** Distancia y duracion estan correlacionadas entre si. Un modelo de regresion multiple debe manejar esto.
3. **Segmentacion necesaria:** Los modelos podrian beneficiarse de entrenarse por separado para aeropuertos vs ciudad.
4. **Variables binarias utiles:** `is_rush_hour` y `is_weekend` pueden aportar informacion incremental como features en modelos predictivos.